# Working with TMol

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/kdidi/sphinx-docs/docs/tutorial/01_working_with_tmol.ipynb)

TMol is a batch-first, PyTorch-native molecular modeling library. This tutorial loads the checked-in 1UBQ mmCIF fixture through a Biotite `AtomArray`, constructs a `PoseStack`, inspects its blocks and metadata, demonstrates PDB export, and uses optional AtomWorks selections.

## Goals

- Choose a PyTorch device explicitly and make runs reproducible.
- Relate `ParameterDatabase`, `PackedBlockTypes`, blocks, and `PoseStack`.
- Load mmCIF and write structures without downloading data.
- Select atoms with masks, AtomWorks queries, and path-style selectors.
- Compare TMol's tensor model with Rosetta's `Pose` and option system.

> **Format recommendation.** Prefer mmCIF/CIF as the primary exchange format. It avoids many PDB limits and can preserve explicit chemical-component bond information that is especially important for ligands. PDB remains useful for compatibility and is shown here only as an export path.

> **GPU optional.** Every cell in this notebook can run on CPU. A CUDA device is selected automatically when available.

## Setup

Imports, reproducibility, device selection, and fixture discovery live under this heading so the documentation can collapse setup details. The input is checked into the repository; no network access is used.

In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlopen

    exec(
        urlopen(
            "https://raw.githubusercontent.com/uw-ipd/tmol/"
            "kdidi/sphinx-docs/docs/tutorial/colab_setup.py"
        ).read(),
        globals(),
    )
    setup_colab(["tmol/tests/data/cif/1UBQ.cif"])

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from biotite.structure import AtomArray
from biotite.structure.io import load_structure
from biotite.structure.io.pdb import PDBFile

import tmol
from tmol.database import ParameterDatabase
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite
from tmol.io.write_pose_stack_pdb import write_pose_stack_pdb

SEED = 20260807
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
repo_root = Path.cwd()
if not (repo_root / "tmol/tests/data/cif/1UBQ.cif").exists():
    repo_root = Path(tmol.__file__).resolve().parents[1]
cif_path = repo_root / "tmol/tests/data/cif/1UBQ.cif"

atom_array = load_structure(
    str(cif_path),
    model=1,
    include_bonds=True,
    extra_fields=["occupancy", "b_factor"],
)
assert isinstance(atom_array, AtomArray)

param_db = ParameterDatabase.get_default()
# Candidate rejection is expected while TMol resolves terminal variants. Keep
# those internal diagnostics out of the tutorial, but reveal them on failure.
pose_diagnostics = StringIO()
try:
    with redirect_stdout(pose_diagnostics), redirect_stderr(pose_diagnostics):
        pose_stack, build_context = pose_stack_from_biotite(
            atom_array,
            torch_device=device,
            param_db=param_db,
            no_optH=True,
            return_context=True,
        )
except Exception:
    print(pose_diagnostics.getvalue())
    raise


def show_table(frame):
    """Use sortable tables in rendered docs, with a pandas fallback."""
    try:
        from itables import show
    except ImportError:
        return display(frame)
    return show(frame)


environment_frame = pd.DataFrame(
    [
        {"component": "TMol", "version": tmol.__version__},
        {"component": "PyTorch", "version": torch.__version__},
        {"component": "device", "version": str(device)},
        {"component": "input", "version": cif_path.name},
    ]
)
show_table(environment_frame)
print(f"input atoms={atom_array.array_length()}")

## From chemistry to coordinates

`ParameterDatabase` is the immutable source of chemical and scoring parameters. A `PackedBlockTypes` object packs the residue types needed by a system onto one device. Each residue-like unit is a **block**, and `PoseStack` stores one or more poses as padded, contiguous tensors over those blocks.

The build context exposes the database, residue-type set, canonical ordering, and packed block types used during Biotite conversion. Reusing a context avoids rebuilding structure-independent chemistry when many compatible structures are loaded. The explicit `no_optH=True` setup choice leaves hydrogens at kinematically ideal positions instead of running hydrogen optimization during I/O; set it to `False` when that preparation step is part of the intended protocol.

In [ ]:
pbt = pose_stack.packed_block_types
real_blocks = pose_stack.block_type_ind64[0] >= 0
block_type_indices = pose_stack.block_type_ind64[0, real_blocks].detach().cpu().tolist()
block_names = [pbt.active_block_types[i].name for i in block_type_indices]

shape_table = pd.DataFrame(
    [
        ("coords", tuple(pose_stack.coords.shape), str(pose_stack.coords.dtype)),
        ("block_coord_offset", tuple(pose_stack.block_coord_offset.shape), str(pose_stack.block_coord_offset.dtype)),
        ("block_type_ind", tuple(pose_stack.block_type_ind.shape), str(pose_stack.block_type_ind.dtype)),
        ("chain_id", tuple(pose_stack.chain_id.shape), str(pose_stack.chain_id.dtype)),
        ("real_atoms", tuple(pose_stack.real_atoms.shape), str(pose_stack.real_atoms.dtype)),
    ],
    columns=["field", "shape", "dtype"],
)
show_table(shape_table)
print(
    f"n_poses={pose_stack.n_poses}, max_n_blocks={pose_stack.max_n_blocks}, "
    f"max_n_pose_atoms={pose_stack.max_n_pose_atoms}"
)
print("first five block types:", block_names[:5])
print("PackedBlockTypes device:", pbt.device)
print(
    "all resolved atom coordinates finite:",
    bool(torch.isfinite(pose_stack.coords[pose_stack.real_atoms]).all()),
)
print("context reuses ParameterDatabase:", build_context.parameter_database is param_db)

**Expected observations.** `coords` has shape `[n_poses, max_n_pose_atoms, 3]`; block-indexed fields have shape `[n_poses, max_n_blocks, ...]`. Padding is represented by sentinel block indices, while `real_atoms` identifies coordinate rows belonging to actual atoms. During construction TMol tests several terminal residue variants against the deposited atom set; rejected candidates are internal resolution attempts, not malformed final atoms. The explicit finite-coordinate check validates the selected result.

`pdb_info` preserves author-facing labels separately from the integer chain and block indices used by kernels.

In [ ]:
residue_frame = pd.DataFrame(
    {
        "block_index": np.arange(pose_stack.max_n_blocks)[real_blocks.cpu().numpy()],
        "chain": pose_stack.pdb_info.chain_labels[0, real_blocks.cpu().numpy()],
        "residue_number": pose_stack.pdb_info.residue_labels[
            0, real_blocks.cpu().numpy()
        ],
        "block_type": block_names,
        "n_atoms": pose_stack.n_ats_per_block[0, real_blocks].detach().cpu().numpy(),
    }
)
show_table(residue_frame)

## PDB compatibility export

The primary input above is mmCIF. TMol currently provides an explicit PDB writer, so this section demonstrates PDB as a compatibility export and then reads it back with Biotite. The output path is intentionally visible so a notebook user can download the generated file.

**Expected observation.** The exported coordinates are finite. Atom counts may differ from the deposited heavy-atom mmCIF because TMol constructs its complete supported residue types, including hydrogens. PDB also cannot carry all mmCIF metadata or reliable ligand bond orders, so compare intended chemistry rather than treating this export as a lossless round trip.

In [ ]:
roundtrip_path = Path("1ubq_tmol_roundtrip.pdb")
# PDB cannot encode every TMol/Biotite annotation; that limitation is already
# explained above, so suppress Biotite's duplicate compatibility warning.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    write_pose_stack_pdb(pose_stack, str(roundtrip_path))
    roundtrip_array = PDBFile.read(str(roundtrip_path)).get_structure(
        model=1,
        include_bonds=True,
        extra_fields=["occupancy", "b_factor"],
    )
print("wrote:", roundtrip_path.resolve())
print("round-trip AtomArray atoms:", roundtrip_array.array_length())
print("finite coordinates:", np.isfinite(roundtrip_array.coord).all())

## AtomWorks mask, query, and path selections

AtomWorks is optional. Its Biotite patch adds `.mask(expression)` and `.query(expression)` to `AtomArray`. The separate path grammar is `CHAIN/RES_NAME/RES_ID/ATOM_NAME/TRANSFORMATION_ID`, with `*` wildcards and bracketed lists or inclusive residue ranges—for example, `A/*/[1-10]/[N,CA,C,O]`.

The guarded cell remains runnable without AtomWorks: it falls back to a plain NumPy mask and reports which optional features were skipped.

In [ ]:
atomworks_available = False
selection_expression = (
    "(chain_id == 'A') & (res_id >= 1) & (res_id <= 10) & "
    "(atom_name in ['N', 'CA', 'C', 'O'])"
)
plain_mask = (
    (atom_array.chain_id == "A")
    & (atom_array.res_id >= 1)
    & (atom_array.res_id <= 10)
    & np.isin(atom_array.atom_name, ["N", "CA", "C", "O"])
)
selection_mask = plain_mask
selected_atoms = atom_array[plain_mask]

atomworks_diagnostics = StringIO()
try:
    with (
        warnings.catch_warnings(),
        redirect_stdout(atomworks_diagnostics),
        redirect_stderr(atomworks_diagnostics),
    ):
        warnings.simplefilter("ignore")
        from atomworks.biotite_patch import monkey_patch_biotite
        from atomworks.io.utils.query import AtomSelectionStack

        monkey_patch_biotite()
        selection_mask = atom_array.mask(selection_expression)
        selected_atoms = atom_array.query(selection_expression)
        path_selection = AtomSelectionStack.from_query(
            "A/*/[1-10]/[N,CA,C,O]"
        )
        path_mask = path_selection.get_mask(atom_array)
        np.testing.assert_array_equal(selection_mask, path_mask)
        atomworks_available = True
except (ImportError, AttributeError):
    print("AtomWorks selection extensions unavailable; using the equivalent NumPy mask.")

print("AtomWorks active:", atomworks_available)
print("selected atoms:", selected_atoms.array_length())
print("selected residues:", np.unique(selected_atoms.res_id))

## Visualize the selection

`tmol.view` is a small PoseStack/AtomArray adapter around the same py3Dmol rendering pattern used by AtomWorks; it is not a second molecular graphics engine. Keeping the adapter lets TMol users pass a `PoseStack` directly, while AtomArray highlighting and hover behavior follow AtomWorks conventions. The gallery renders each selection through py3Dmol's native notebook HTML path so it also works in static Sphinx output.

In [ ]:
sidechain_mask = (
    (atom_array.chain_id == "A")
    & (atom_array.res_id >= 1)
    & (atom_array.res_id <= 10)
    & ~np.isin(atom_array.atom_name, ["N", "CA", "C", "O", "OXT"])
)
try:
    selection_viewer = tmol.view(
        atom_array,
        highlighted=selection_mask,
        width=720,
        height=420,
    )
    selection_viewer.show()
    display(
        tmol.selection_gallery(
            atom_array,
            {
                "Backbone, residues 1–10": selection_mask,
                "Side chains, residues 1–10": sidechain_mask,
                "All Cα atoms": atom_array.atom_name == "CA",
            },
        )
    )
except ImportError as exc:
    print("Selection viewer unavailable in this environment:", exc)

## Rosetta comparison

A Rosetta `Pose` is a single rich object with residues, conformation, energies, and attached metadata. A TMol `PoseStack` is deliberately batch-first: chemistry/connectivity metadata is block-indexed, while Cartesian coordinates are contiguous PyTorch tensors. One TMol block is closest to a residue-like chemical unit, but not every block must be a canonical amino acid.

Rosetta applications often obtain behavior through a process-wide flags/options system. TMol has no global Rosetta-style flags layer: device, parameter database, score-function options, I/O choices, and protocol settings are explicit Python arguments or object configuration. This is more verbose but makes notebook state and batching assumptions visible.

See the official full tutorials for [Working with Rosetta](https://docs.rosettacommons.org/demos/latest/tutorials/Working_With_Rosetta/working_with_rosetta), [input and output](https://docs.rosettacommons.org/demos/latest/tutorials/input_and_output/input_and_output), [core concepts](https://docs.rosettacommons.org/demos/latest/tutorials/Core_Concepts/Core_Concepts), [commonly used options](https://docs.rosettacommons.org/demos/latest/tutorials/commonly_used_options/commonly_used_options), and [PyRosetta Pose basics](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/02.01-Pose-Basics.ipynb).

## Limitations

- Prefer mmCIF/CIF for input and archival exchange. PDB cannot represent every mmCIF annotation and does not reliably preserve ligand bond order; use CIF with explicit chemical-component bonds, MOL2, or prepared `.tmol` chemistry for ligands.
- TMol's current convenience writer targets PDB, so that export is compatibility-oriented rather than a lossless mmCIF round trip.
- The default database covers TMol-supported chemistry, not every Rosetta residue type.
- AtomWorks is optional and is used only for ergonomic selection; NumPy masks remain valid Biotite selections.
- The highlighted-`AtomArray` viewer call depends on the parent documentation helper. The core `tmol.view(PoseStack)` API remains available without AtomWorks.
- `PoseStack` batches share a device and compatible chemical database; arbitrary objects cannot be concatenated without reconciling chemistry.

## Exercises

1. Change the device selection to force CPU and confirm all tensor devices agree.
2. Select residues 20–30 with both an AtomWorks expression and a path selector; assert that their masks match.
3. Summarize the number of atoms per block type and identify terminal variants.
4. Reuse `build_context` to construct a second compatible `PoseStack` from `roundtrip_array`.
5. Write the round-tripped pose under a new filename and compare author residue labels rather than raw PDB text.

## References

- [TMol repository](https://github.com/uw-ipd/tmol)
- [Biotite structure documentation](https://www.biotite-python.org/latest/apidoc/biotite.structure.html)
- [AtomWorks selection syntax and interactive examples](https://baker-laboratory.github.io/atomworks-dev/latest/io/utils/selection_syntax.html#id2)
- [Rosetta: Working with Rosetta](https://docs.rosettacommons.org/demos/latest/tutorials/Working_With_Rosetta/working_with_rosetta)
- [Rosetta input/output tutorial](https://docs.rosettacommons.org/demos/latest/tutorials/input_and_output/input_and_output)
- [PyRosetta Pose basics](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/02.01-Pose-Basics.ipynb)